# Part 3: Neural Network with PyTorch – CIFAR-10

## Overview
In this notebook, we implement neural networks using PyTorch at three levels of abstraction to classify CIFAR-10 images. We then design an improved network (Part V) to achieve higher accuracy.

**Dataset:** CIFAR-10 – 60,000 color images (32×32) across 10 classes  
**Framework:** PyTorch with GPU acceleration (CUDA)

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.data import sampler
import torchvision.datasets as dset
import torchvision.transforms as T
import numpy as np

USE_GPU = True
dtype = torch.float32

if USE_GPU and torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print_every = 100
print('using device:', device)

using device: cuda


In [2]:
# Reproducibility: set a fixed seed so results are repeatable and match the report
import os, random
SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # Make CUDA/cuDNN as deterministic as possible
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # Some ops may not have deterministic implementations on all devices/versions
    torch.use_deterministic_algorithms(True, warn_only=True)

set_seed(SEED)

CHECKPOINT_DIR = "./checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Table of Contents

This assignment has 5 parts. You will learn PyTorch on **three different levels of abstraction**, which will help you understand it better and prepare you for the final project.

1. Part I, Preparation: we will use CIFAR-10 dataset.
2. Part II, Barebones PyTorch: **Abstraction level 1**, we will work directly with the lowest-level PyTorch Tensors.
3. Part III, PyTorch Module API: **Abstraction level 2**, we will use `nn.Module` to define arbitrary neural network architecture.
4. Part IV, PyTorch Sequential API: **Abstraction level 3**, we will use `nn.Sequential` to define a linear feed-forward network very conveniently.
5. Part V, CIFAR-10 open-ended challenge: please implement your own network to get as high accuracy as possible on CIFAR-10. You can experiment with any layer, optimizer, hyperparameters or other advanced features.

Here is a table of comparison:

| API           | Flexibility | Convenience |
|---------------|-------------|-------------|
| Barebone      | High        | Low         |
| `nn.Module`     | High        | Medium      |
| `nn.Sequential` | Low         | High        |

# GPU

You can manually switch to a GPU device on Colab by clicking `Runtime -> Change runtime type` and selecting `GPU` under `Hardware Accelerator`. You should do this before running the following cells to import packages, since the kernel gets restarted upon switching runtimes.

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.data import sampler

import torchvision.datasets as dset
import torchvision.transforms as T

import numpy as np

USE_GPU = True
dtype = torch.float32 # We will be using float throughout this assignment.

if USE_GPU and torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

# Constant to control how frequently we print train loss.
print_every = 100
print('using device:', device)

using device: cuda


# Part I. Preparation

Now, let's load the CIFAR-10 dataset. This might take a couple of minutes the first
time you run the code, but the files should stay cached afterwards.

In earlier sections of this assignment, dataset handling was performed manually.
Here, we use PyTorch's built-in utilities to load, preprocess, and iterate over
the data in minibatches.


In [4]:
NUM_TRAIN = 49000

transform = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465),
                (0.2023, 0.1994, 0.2010))
])

cifar10_train = dset.CIFAR10('./cifar10', train=True, download=True,
                             transform=transform)
g = torch.Generator()
g.manual_seed(SEED)

loader_train = DataLoader(cifar10_train, batch_size=64,
                          sampler=sampler.SubsetRandomSampler(range(NUM_TRAIN), generator=g))

cifar10_val = dset.CIFAR10('./cifar10', train=True, download=True,
                           transform=transform)
loader_val = DataLoader(cifar10_val, batch_size=64,
                        sampler=sampler.SubsetRandomSampler(range(NUM_TRAIN, 50000), generator=g))

cifar10_test = dset.CIFAR10('./cifar10', train=False, download=True,
                            transform=transform)
loader_test = DataLoader(cifar10_test, batch_size=64)

print("Data loaded successfully!")
print(f"Train batches: {len(loader_train)}")
print(f"Val batches: {len(loader_val)}")
print(f"Test batches: {len(loader_test)}")


100%|██████████| 170M/170M [00:13<00:00, 12.9MB/s]


Data loaded successfully!
Train batches: 766
Val batches: 16
Test batches: 157


## 3.1 Barebones PyTorch
We implement a two-layer fully-connected network manually using raw PyTorch tensors. Weights are initialized using Kaiming normalization and updated manually via SGD.

# Part II. Barebones PyTorch

PyTorch ships with high-level APIs to help us define model architectures conveniently, which we will cover in Part II of this assignment. In this section, we will start with the barebone PyTorch elements to understand the autograd engine better. After this exercise, you will come to appreciate the high-level model API more.

We will start with a simple fully-connected ReLU network with two hidden layers and no biases for CIFAR classification.
This implementation computes the forward pass using operations on PyTorch Tensors, and uses PyTorch autograd to compute gradients. It is important that you understand every line, because you will write a harder version after the example.

When we create a PyTorch Tensor with `requires_grad=True`, then operations involving that Tensor will not just compute values; they will also build up a computational graph in the background, allowing us to easily backpropagate through the graph to compute gradients of some Tensors with respect to a downstream loss. Concretely if x is a Tensor with `x.requires_grad == True` then after backpropagation `x.grad` will be another Tensor holding the gradient of x with respect to the scalar loss at the end.

### PyTorch Tensors: Flatten Function
A PyTorch Tensor is conceptionally similar to a numpy array: it is an n-dimensional grid of numbers, and like numpy PyTorch provides many functions to efficiently operate on Tensors. As a simple example, we provide a `flatten` function below which reshapes image data for use in a fully-connected neural network.

Recall that image data is typically stored in a Tensor of shape N x C x H x W, where:

* N is the number of datapoints
* C is the number of channels
* H is the height of the intermediate feature map in pixels
* W is the height of the intermediate feature map in pixels

When we use fully connected affine layers to process the image, however, we want each datapoint to be represented by a single vector -- it's no longer useful to segregate the different channels, rows, and columns of the data. So, we use a "flatten" operation to collapse the `C x H x W` values per representation into a single long vector. The flatten function below first reads in the N, C, H, and W values from a given batch of data, and then returns a "view" of that data. "View" is analogous to numpy's "reshape" method: it reshapes x's dimensions to be N x ??, where ?? is allowed to be anything (in this case, it will be C x H x W, but we don't need to specify that explicitly).

In [5]:
def flatten(x):
    N = x.shape[0] # read in N, C, H, W
    return x.view(N, -1)  # "flatten" the C * H * W values into a single vector per image

def test_flatten():
    x = torch.arange(12).view(2, 1, 3, 2)
    print('Before flattening: ', x)
    print('After flattening: ', flatten(x))

test_flatten()

Before flattening:  tensor([[[[ 0,  1],
          [ 2,  3],
          [ 4,  5]]],


        [[[ 6,  7],
          [ 8,  9],
          [10, 11]]]])
After flattening:  tensor([[ 0,  1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10, 11]])


### Barebones PyTorch: Two-Layer Network

Here we define a function `two_layer_fc` which performs the forward pass of a two-layer fully-connected ReLU network on a batch of image data. After defining the forward pass we check that it doesn't crash and that it produces outputs of the right shape by running zeros through the network.

You don't have to write any code here, but it's important that you read and understand the implementation.

In [6]:
import torch.nn.functional as F  # useful stateless functions

def two_layer_fc(x, params):
    """
    A fully-connected neural network; the architecture is:
    NN is fully connected -> ReLU -> fully connected layer.
    Note that this function only defines the forward pass;
    PyTorch will take care of the backward pass for us.

    The input to the network will be a minibatch of data, of shape
    (N, d1, ..., dM) where d1 * ... * dM = D. The hidden layer will have H units,
    and the output layer will produce scores for C classes.

    Inputs:
    - x: A PyTorch Tensor of shape (N, d1, ..., dM) giving a minibatch of
      input data.
    - params: A list [w1, w2] of PyTorch Tensors giving weights for the network;
      w1 has shape (D, H) and w2 has shape (H, C).

    Returns:
    - scores: A PyTorch Tensor of shape (N, C) giving classification scores for
      the input data x.
    """
    # first we flatten the image
    x = flatten(x)  # shape: [batch_size, C x H x W]

    w1, w2 = params

    # Forward pass: compute predicted y using operations on Tensors. Since w1 and
    # w2 have requires_grad=True, operations involving these Tensors will cause
    # PyTorch to build a computational graph, allowing automatic computation of
    # gradients. Since we are no longer implementing the backward pass by hand we
    # don't need to keep references to intermediate values.
    # you can also use `.clamp(min=0)`, equivalent to F.relu()
    x = F.relu(x.mm(w1))
    x = x.mm(w2)
    return x


def two_layer_fc_test():
    hidden_layer_size = 42
    x = torch.zeros((64, 50), dtype=dtype)  # minibatch size 64, feature dimension 50
    w1 = torch.zeros((50, hidden_layer_size), dtype=dtype)
    w2 = torch.zeros((hidden_layer_size, 10), dtype=dtype)
    scores = two_layer_fc(x, [w1, w2])
    print(scores.size())  # you should see [64, 10]

two_layer_fc_test()

torch.Size([64, 10])


### Barebones PyTorch: Initialization
Let's write a couple utility methods to initialize the weight matrices for our models.

- `random_weight(shape)` initializes a weight tensor with the Kaiming normalization method.
- `zero_weight(shape)` initializes a weight tensor with all zeros. Useful for instantiating bias parameters.

The `random_weight` function uses the Kaiming normal initialization method, described in:

He et al, *Delving Deep into Rectifiers: Surpassing Human-Level Performance on ImageNet Classification*, ICCV 2015, https://arxiv.org/abs/1502.01852

In [7]:
def random_weight(shape):
    """
    Create random Tensors for weights; setting requires_grad=True means that we
    want to compute gradients for these Tensors during the backward pass.
    We use Kaiming normalization: sqrt(2 / fan_in)
    """
    if len(shape) == 2:  # FC weight
        fan_in = shape[0]
    else:
        fan_in = np.prod(shape[1:]) # general fan-in computation
    # randn is standard normal distribution generator.
    w = torch.randn(shape, device=device, dtype=dtype) * np.sqrt(2. / fan_in)
    w.requires_grad = True
    return w

def zero_weight(shape):
    return torch.zeros(shape, device=device, dtype=dtype, requires_grad=True)

# create a weight of shape [3 x 5]
# you should see the type `torch.cuda.FloatTensor` if you use GPU.
# Otherwise it should be `torch.FloatTensor`
random_weight((3, 5))

tensor([[ 0.1584,  1.7648, -0.1405,  0.6933, -1.5713],
        [ 0.5332, -0.5303, -0.6675,  0.4311, -1.0413],
        [-1.3571, -0.2477, -0.0756,  0.1627, -0.9148]], device='cuda:0',
       requires_grad=True)

### Barebones PyTorch: Check Accuracy
When training the model we will use the following function to check the accuracy of our model on the training or validation sets.

When checking accuracy we don't need to compute any gradients; as a result we don't need PyTorch to build a computational graph for us when we compute scores. To prevent a graph from being built we scope our computation under a `torch.no_grad()` context manager.

In [8]:
def check_accuracy_part2(loader, model_fn, params):
    """
    Check the accuracy of a classification model.

    Inputs:
    - loader: A DataLoader for the data split we want to check
    - model_fn: A function that performs the forward pass of the model,
      with the signature scores = model_fn(x, params)
    - params: List of PyTorch Tensors giving parameters of the model

    Returns: Nothing, but prints the accuracy of the model
    """
    split = 'val' if loader.dataset.train else 'test'
    print('Checking accuracy on the %s set' % split)
    num_correct, num_samples = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device, dtype=dtype)  # move to device, e.g. GPU
            y = y.to(device=device, dtype=torch.int64)
            scores = model_fn(x, params)
            _, preds = scores.max(1)
            num_correct += (preds == y).sum()
            num_samples += preds.size(0)
        acc = float(num_correct) / num_samples
        print('Got %d / %d correct (%.2f%%)' % (num_correct, num_samples, 100 * acc))

### BareBones PyTorch: Training Loop
We can now set up a basic training loop to train our network. We will train the model using stochastic gradient descent without momentum. We will use `torch.functional.cross_entropy` to compute the loss; you can [read about it here](http://pytorch.org/docs/stable/nn.html#cross-entropy).

The training loop takes as input the neural network function, a list of initialized parameters (`[w1, w2]` in our example), and learning rate.

In [9]:
def train_part2(model_fn, params, learning_rate):
    """
    Train a model on CIFAR-10.

    Inputs:
    - model_fn: A Python function that performs the forward pass of the model.
      It should have the signature scores = model_fn(x, params) where x is a
      PyTorch Tensor of image data, params is a list of PyTorch Tensors giving
      model weights, and scores is a PyTorch Tensor of shape (N, C) giving
      scores for the elements in x.
    - params: List of PyTorch Tensors giving weights for the model
    - learning_rate: Python scalar giving the learning rate to use for SGD

    Returns: Nothing
    """
    for t, (x, y) in enumerate(loader_train):
        # Move the data to the proper device (GPU or CPU)
        x = x.to(device=device, dtype=dtype)
        y = y.to(device=device, dtype=torch.long)

        # Forward pass: compute scores and loss
        scores = model_fn(x, params)
        loss = F.cross_entropy(scores, y)

        # Backward pass: PyTorch figures out which Tensors in the computational
        # graph has requires_grad=True and uses backpropagation to compute the
        # gradient of the loss with respect to these Tensors, and stores the
        # gradients in the .grad attribute of each Tensor.
        loss.backward()

        # Update parameters. We don't want to backpropagate through the
        # parameter updates, so we scope the updates under a torch.no_grad()
        # context manager to prevent a computational graph from being built.
        with torch.no_grad():
            for w in params:
                w -= learning_rate * w.grad

                # Manually zero the gradients after running the backward pass
                w.grad.zero_()

        if t % print_every == 0:
            print('Iteration %d, loss = %.4f' % (t, loss.item()))
            check_accuracy_part2(loader_val, model_fn, params)
            print()

### BareBones PyTorch: Train a Two-Layer Network
Now we are ready to run the training loop. We need to explicitly allocate tensors for the fully connected weights, `w1` and `w2`.

Each minibatch of CIFAR has 64 examples, so the tensor shape is `[64, 3, 32, 32]`.

After flattening, `x` shape should be `[64, 3 * 32 * 32]`. This will be the size of the first dimension of `w1`.
The second dimension of `w1` is the hidden layer size, which will also be the first dimension of `w2`.

Finally, the output of the network is a 10-dimensional vector that represents the probability distribution over 10 classes.

You do not need to tune any hyperparameters. After training for one epoch, you
should observe that the model learns non-trivial structure in the data.


In [10]:
hidden_layer_size = 4000
learning_rate = 1e-2

w1 = random_weight((3 * 32 * 32, hidden_layer_size))
w2 = random_weight((hidden_layer_size, 10))

train_part2(two_layer_fc, [w1, w2], learning_rate)

Iteration 0, loss = 3.7298
Checking accuracy on the val set
Got 156 / 1000 correct (15.60%)

Iteration 100, loss = 2.1409
Checking accuracy on the val set
Got 307 / 1000 correct (30.70%)

Iteration 200, loss = 2.1455
Checking accuracy on the val set
Got 371 / 1000 correct (37.10%)

Iteration 300, loss = 1.9368
Checking accuracy on the val set
Got 374 / 1000 correct (37.40%)

Iteration 400, loss = 1.7113
Checking accuracy on the val set
Got 444 / 1000 correct (44.40%)

Iteration 500, loss = 1.5834
Checking accuracy on the val set
Got 425 / 1000 correct (42.50%)

Iteration 600, loss = 1.6879
Checking accuracy on the val set
Got 425 / 1000 correct (42.50%)

Iteration 700, loss = 1.5951
Checking accuracy on the val set
Got 442 / 1000 correct (44.20%)



## 3.2 Module API
We reimplement the same two-layer network using nn.Module. This approach tracks parameters automatically and uses PyTorch's built-in optimizers.

# Part III. PyTorch Module API

Barebone PyTorch requires that we track all the parameter tensors by hand. This is fine for small networks with a few tensors, but it would be extremely inconvenient and error-prone to track tens or hundreds of tensors in larger networks.

PyTorch provides the `nn.Module` API for you to define arbitrary network architectures, while tracking every learnable parameters for you. In Part II, we implemented SGD ourselves. PyTorch also provides the `torch.optim` package that implements all the common optimizers, such as RMSProp, Adagrad, and Adam. It even supports approximate second-order methods like L-BFGS! You can refer to the [doc](http://pytorch.org/docs/master/optim.html) for the exact specifications of each optimizer.

To use the Module API, follow the steps below:

1. Subclass `nn.Module`. Give your network class an intuitive name like `TwoLayerFC`.

2. In the constructor `__init__()`, define all the layers you need as class attributes. Layer objects like `nn.Linear` are themselves `nn.Module` subclasses and contain learnable parameters, so that you don't have to instantiate the raw tensors yourself. `nn.Module` will track these internal parameters for you. Refer to the [doc](http://pytorch.org/docs/master/nn.html) to learn more about the dozens of builtin layers. **Warning**: don't forget to call the `super().__init__()` first!

3. In the `forward()` method, define the *connectivity* of your network. You should use the attributes defined in `__init__` as function calls that take tensor as input and output the "transformed" tensor. Do *not* create any new layers with learnable parameters in `forward()`! All of them must be declared upfront in `__init__`.

After you define your Module subclass, you can instantiate it as an object and call it just like the NN forward function in part II.

### Module API: Two-Layer Network
Here is a concrete example of a 2-layer fully connected network:

In [11]:
class TwoLayerFC(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        # assign layer objects to class attributes
        self.fc1 = nn.Linear(input_size, hidden_size)
        # nn.init package contains convenient initialization methods
        # http://pytorch.org/docs/master/nn.html#torch-nn-init
        nn.init.kaiming_normal_(self.fc1.weight)
        self.fc2 = nn.Linear(hidden_size, num_classes)
        nn.init.kaiming_normal_(self.fc2.weight)

    def forward(self, x):
        # forward always defines connectivity
        x = flatten(x)
        scores = self.fc2(F.relu(self.fc1(x)))
        return scores

def test_TwoLayerFC():
    input_size = 50
    x = torch.zeros((64, input_size), dtype=dtype)  # minibatch size 64, feature dimension 50
    model = TwoLayerFC(input_size, 42, 10)
    scores = model(x)
    print(scores.size())  # you should see [64, 10]
test_TwoLayerFC()

torch.Size([64, 10])


### Module API: Check Accuracy
Given the validation or test set, we can check the classification accuracy of a neural network.

This version is slightly different from the one in part II. You don't manually pass in the parameters anymore.

In [12]:
def check_accuracy_part34(loader, model):
    if loader.dataset.train:
        print('Checking accuracy on validation set')
    else:
        print('Checking accuracy on test set')
    num_correct = 0
    num_samples = 0
    model.eval()  # set model to evaluation mode
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device, dtype=dtype)  # move to device, e.g. GPU
            y = y.to(device=device, dtype=torch.long)
            scores = model(x)
            _, preds = scores.max(1)
            num_correct += (preds == y).sum()
            num_samples += preds.size(0)
        acc = float(num_correct) / num_samples
        print('Got %d / %d correct (%.2f)' % (num_correct, num_samples, 100 * acc))
    return acc


In [13]:
def get_accuracy(loader, model):
    """Compute accuracy (0..1) without printing."""
    num_correct = 0
    num_samples = 0
    model.eval()
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device, dtype=dtype)
            y = y.to(device=device, dtype=torch.long)
            scores = model(x)
            _, preds = scores.max(1)
            num_correct += (preds == y).sum()
            num_samples += preds.size(0)
    return float(num_correct) / num_samples

### Module API: Training Loop
We also use a slightly different training loop. Rather than updating the values of the weights ourselves, we use an Optimizer object from the `torch.optim` package, which abstract the notion of an optimization algorithm and provides implementations of most of the algorithms commonly used to optimize neural networks.

In [14]:
import copy

def train_part34(model, optimizer, epochs=1, *, checkpoint_path=None):
    """
    Train a model on CIFAR-10 (PyTorch Module API) and keep the best checkpoint
    according to validation accuracy.

    Returns:
    - best_val_acc: Best validation accuracy (0..1) observed during training.
    """
    model = model.to(device=device)
    best_val_acc = -1.0
    best_state = None

    for epoch in range(epochs):
        model.train()
        for t, (x, y) in enumerate(loader_train):
            x = x.to(device=device, dtype=dtype)
            y = y.to(device=device, dtype=torch.long)

            scores = model(x)
            loss = F.cross_entropy(scores, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if t % print_every == 0:
                # Light progress print (validation check without changing the model)
                val_acc = get_accuracy(loader_val, model)
                print(f"Epoch {epoch+1}/{epochs} | Iteration {t} | loss={loss.item():.4f} | val_acc={100*val_acc:.2f}%")

        # End-of-epoch validation + best checkpointing
        val_acc = get_accuracy(loader_val, model)
        print(f"End of epoch {epoch+1}: val_acc={100*val_acc:.2f}%")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

    # Restore best weights
    if best_state is not None:
        model.load_state_dict(best_state)

    # Optional: save best model checkpoint to allow exact reproduction without retraining
    if checkpoint_path is not None:
        torch.save({
            "seed": SEED,
            "best_val_acc": best_val_acc,
            "state_dict": model.state_dict(),
        }, checkpoint_path)

    return best_val_acc

### Module API: Train a Two-Layer Network
Now we are ready to run the training loop. In contrast to part II, we don't explicitly allocate parameter tensors anymore.

Simply pass the input size, hidden layer size, and number of classes (i.e. output size) to the constructor of `TwoLayerFC`.

You also need to define an optimizer that tracks all the learnable parameters inside `TwoLayerFC`.

You do not need to tune any hyperparameters. After training for one epoch, you
should observe that the model learns non-trivial structure in the data.


In [15]:
hidden_layer_size = 4000
learning_rate = 1e-2
model = TwoLayerFC(3 * 32 * 32, hidden_layer_size, 10)
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

train_part34(model, optimizer)

Epoch 1/1 | Iteration 0 | loss=3.9061 | val_acc=14.90%
Epoch 1/1 | Iteration 100 | loss=2.2133 | val_acc=30.30%
Epoch 1/1 | Iteration 200 | loss=2.1717 | val_acc=32.80%
Epoch 1/1 | Iteration 300 | loss=1.7208 | val_acc=39.80%
Epoch 1/1 | Iteration 400 | loss=1.7460 | val_acc=44.60%
Epoch 1/1 | Iteration 500 | loss=1.9113 | val_acc=42.20%
Epoch 1/1 | Iteration 600 | loss=1.8127 | val_acc=42.80%
Epoch 1/1 | Iteration 700 | loss=1.7619 | val_acc=43.40%
End of epoch 1: val_acc=38.40%


0.384

## 3.3 Sequential API
We use nn.Sequential for a cleaner, more concise definition of the same network. We add Nesterov momentum to the SGD optimizer for faster convergence.

# Part IV. PyTorch Sequential API

Part III introduced the PyTorch Module API, which allows you to define arbitrary learnable layers and their connectivity.

For simple models like a stack of feed forward layers, you still need to go through 3 steps: subclass `nn.Module`, assign layers to class attributes in `__init__`, and call each layer one by one in `forward()`. Is there a more convenient way?

Fortunately, PyTorch provides a container Module called `nn.Sequential`, which merges the above steps into one. It is not as flexible as `nn.Module`, because you cannot specify more complex topology than a feed-forward stack, but it's good enough for many use cases.

### Sequential API: Two-Layer Network
Let's see how to rewrite our two-layer fully connected network example with `nn.Sequential`, and train it using the training loop defined above.

Again, you do not need to tune any hyperparameters. After one epoch of training,
you should observe that the model learns non-trivial structure in the data.


In [16]:
# We need to wrap `flatten` function in a module in order to stack it
# in nn.Sequential
class Flatten(nn.Module):
    def forward(self, x):
        return flatten(x)

hidden_layer_size = 4000
learning_rate = 1e-2

model = nn.Sequential(
    Flatten(),
    nn.Linear(3 * 32 * 32, hidden_layer_size),
    nn.ReLU(),
    nn.Linear(hidden_layer_size, 10),
)

# you can use Nesterov momentum in optim.SGD
optimizer = optim.SGD(model.parameters(), lr=learning_rate,
                     momentum=0.9, nesterov=True)

train_part34(model, optimizer)

Epoch 1/1 | Iteration 0 | loss=2.3710 | val_acc=14.60%
Epoch 1/1 | Iteration 100 | loss=1.7858 | val_acc=40.20%
Epoch 1/1 | Iteration 200 | loss=1.7408 | val_acc=41.80%
Epoch 1/1 | Iteration 300 | loss=1.7588 | val_acc=43.90%
Epoch 1/1 | Iteration 400 | loss=1.5386 | val_acc=43.40%
Epoch 1/1 | Iteration 500 | loss=1.6603 | val_acc=42.90%
Epoch 1/1 | Iteration 600 | loss=1.5491 | val_acc=41.40%
Epoch 1/1 | Iteration 700 | loss=2.1080 | val_acc=42.40%
End of epoch 1: val_acc=44.90%


0.449

In [17]:
# Hyperparameter Search - comparing different architectures
configs = [
    {'name': '2-layer small',  'hidden': 512,  'layers': 2, 'dropout': 0.3, 'lr': 1e-3},
    {'name': '2-layer medium', 'hidden': 1024, 'layers': 2, 'dropout': 0.3, 'lr': 1e-3},
    {'name': '3-layer medium', 'hidden': 1024, 'layers': 3, 'dropout': 0.3, 'lr': 1e-3},
    {'name': '4-layer large',  'hidden': 2048, 'layers': 4, 'dropout': 0.3, 'lr': 1e-3},
    {'name': '4-layer + low dropout', 'hidden': 2048, 'layers': 4, 'dropout': 0.2, 'lr': 1e-3},
]

print(f"{'Config':<25} {'Val Accuracy':>12}")
print("-" * 40)

search_results = []

for cfg in configs:
    # Build model dynamically
    layers = [Flatten()]
    in_size = 3 * 32 * 32
    for _ in range(cfg['layers']):
        layers += [
            nn.Linear(in_size, cfg['hidden']),
            nn.BatchNorm1d(cfg['hidden']),
            nn.ReLU(),
            nn.Dropout(cfg['dropout'])
        ]
        in_size = cfg['hidden']
    layers.append(nn.Linear(in_size, 10))

    model_tmp = nn.Sequential(*layers)
    optimizer_tmp = optim.Adam(model_tmp.parameters(), lr=cfg['lr'])
    train_part34(model_tmp, optimizer_tmp, epochs=3)

    # Check val accuracy
    model_tmp.eval()
    num_correct, num_samples = 0, 0
    with torch.no_grad():
        for x, y in loader_val:
            x = x.to(device=device, dtype=dtype)
            y = y.to(device=device, dtype=torch.long)
            scores = model_tmp(x)
            _, preds = scores.max(1)
            num_correct += (preds == y).sum()
            num_samples += preds.size(0)
    acc = float(num_correct) / num_samples
    print(f"{cfg['name']:<25} {acc:>12.3f}")
    search_results.append((acc, cfg))

best_cfg_search = max(search_results, key=lambda x: x[0])[1]
print(f"\nBest config: {best_cfg_search['name']}")

Config                    Val Accuracy
----------------------------------------
Epoch 1/3 | Iteration 0 | loss=2.4011 | val_acc=16.80%
Epoch 1/3 | Iteration 100 | loss=1.6269 | val_acc=36.50%
Epoch 1/3 | Iteration 200 | loss=1.6313 | val_acc=42.60%
Epoch 1/3 | Iteration 300 | loss=1.8126 | val_acc=40.20%
Epoch 1/3 | Iteration 400 | loss=1.6050 | val_acc=41.80%
Epoch 1/3 | Iteration 500 | loss=1.8207 | val_acc=45.80%
Epoch 1/3 | Iteration 600 | loss=1.6307 | val_acc=44.80%
Epoch 1/3 | Iteration 700 | loss=1.7740 | val_acc=46.80%
End of epoch 1: val_acc=47.00%
Epoch 2/3 | Iteration 0 | loss=1.9921 | val_acc=45.60%
Epoch 2/3 | Iteration 100 | loss=1.6593 | val_acc=47.30%
Epoch 2/3 | Iteration 200 | loss=1.2613 | val_acc=46.60%
Epoch 2/3 | Iteration 300 | loss=1.5281 | val_acc=48.10%
Epoch 2/3 | Iteration 400 | loss=1.4921 | val_acc=48.40%
Epoch 2/3 | Iteration 500 | loss=1.5516 | val_acc=48.80%
Epoch 2/3 | Iteration 600 | loss=1.3617 | val_acc=50.00%
Epoch 2/3 | Iteration 700 | loss=1.382

## 3.4 Open-Ended Challenge – BetterNet
We design a deeper network with Batch Normalization and Dropout to improve accuracy on CIFAR-10. The architecture has 4 fully-connected layers with ReLU activations, trained using the Adam optimizer for 10 epochs.

**Architecture:**
- FC(3072 → 2048) → BatchNorm → ReLU → Dropout(0.3)
- FC(2048 → 1024) → BatchNorm → ReLU → Dropout(0.3)
- FC(1024 → 512) → BatchNorm → ReLU → Dropout(0.3)
- FC(512 → 10)

**Optimizer:** Adam (lr=1e-3, weight_decay=1e-4)

# Part V. CIFAR-10 open-ended challenge

In this section, you can experiment with whatever architecture you'd like on CIFAR-10.

Now it's your job to experiment with architectures, hyperparameters, loss functions, and optimizers to train a model that achieves high accuracy on the CIFAR-10 **validation** set within 10 epochs. You can use the check_accuracy and train functions from above. You can use either `nn.Module` or `nn.Sequential` API.

Describe what you did at Overleaf report.

* Layers in torch.nn package: http://pytorch.org/docs/stable/nn.html
* Activations: http://pytorch.org/docs/stable/nn.html#non-linear-activations
* Loss functions: http://pytorch.org/docs/stable/nn.html#loss-functions
* Optimizers: http://pytorch.org/docs/stable/optim.html


### Things you might try:
- **Batch normalization**: Try adding batch normalization after fully-connected
  layers. Does training become more stable or faster?
- **Network architecture**: The network above has two layers of trainable parameters. Can you do better with a deep network?
- **Regularization**: Add l2 weight regularization, or perhaps use Dropout.

### Tips for training
For each network architecture that you try, you should tune the learning rate and other hyperparameters. When doing this there are a couple important things to keep in mind:

- If the parameters are working well, you should see improvement within a few hundred iterations
- Remember the coarse-to-fine approach for hyperparameter tuning: start by testing a large range of hyperparameters for just a few training iterations to find the combinations of parameters that are working at all.
- Once you have found some sets of parameters that seem to work, search more finely around these parameters. You may need to train for more epochs.
- You should use the validation set for hyperparameter search, and save your test set for evaluating your architecture on the best parameters as selected by the validation set.

### Going above and beyond
If you are feeling adventurous there are many other features you can implement to try and improve your performance. You are **not required** to implement any of these, but don't miss the fun if you have time!

- Alternative optimizers: you can try Adam, Adagrad, RMSprop, etc.
- Alternative activation functions such as leaky ReLU, parametric ReLU, ELU, or MaxOut.
- Model ensembles
- Data augmentation
- New Architectures
  - [ResNets](https://arxiv.org/abs/1512.03385) where the input from the previous layer is added to the output.
  - [DenseNets](https://arxiv.org/abs/1608.06993) where inputs into previous layers are concatenated together.
  - [This blog has an in-depth overview](https://chatbotslife.com/resnets-highwaynets-and-densenets-oh-my-9bb15918ee32)
  - [LeNet-5](http://yann.lecun.com/exdb/publis/pdf/lecun-98.pdf) where convolutional layers and pooling are used to exploit spatial structure in images.
  - [AlexNet](https://papers.nips.cc/paper/2012/file/c399862d3b9d6b76c8436e924a68c45b-Paper.pdf) which demonstrated the effectiveness of deep convolutional neural networks for large-scale image classification.
  - [This blog provides an accessible introduction to CNNs](https://cs231n.github.io/convolutional-networks/) explaining convolution, pooling, and why CNNs outperform fully-connected networks on image data.


### Have fun and happy training!

In [18]:
class BetterNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3 * 32 * 32, 2048)
        nn.init.kaiming_normal_(self.fc1.weight)
        self.bn1 = nn.BatchNorm1d(2048)

        self.fc2 = nn.Linear(2048, 1024)
        nn.init.kaiming_normal_(self.fc2.weight)
        self.bn2 = nn.BatchNorm1d(1024)

        self.fc3 = nn.Linear(1024, 512)
        nn.init.kaiming_normal_(self.fc3.weight)
        self.bn3 = nn.BatchNorm1d(512)

        self.fc4 = nn.Linear(512, 10)
        nn.init.kaiming_normal_(self.fc4.weight)

        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = flatten(x)
        x = self.dropout(F.relu(self.bn1(self.fc1(x))))
        x = self.dropout(F.relu(self.bn2(self.fc2(x))))
        x = self.dropout(F.relu(self.bn3(self.fc3(x))))
        x = self.fc4(x)
        return x

model = BetterNet()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

ckpt_path = os.path.join(CHECKPOINT_DIR, f"betternet_seed{SEED}.pth")
best_val_acc = train_part34(model, optimizer, epochs=10, checkpoint_path=ckpt_path)
print(f"Best validation accuracy: {100*best_val_acc:.2f}%")
print("Evaluating best model on test set:")
check_accuracy_part34(loader_test, model)


Epoch 1/10 | Iteration 0 | loss=2.8644 | val_acc=17.50%
Epoch 1/10 | Iteration 100 | loss=1.8364 | val_acc=35.50%
Epoch 1/10 | Iteration 200 | loss=1.7213 | val_acc=42.40%
Epoch 1/10 | Iteration 300 | loss=1.7909 | val_acc=41.00%
Epoch 1/10 | Iteration 400 | loss=1.4418 | val_acc=42.60%
Epoch 1/10 | Iteration 500 | loss=1.6795 | val_acc=39.70%
Epoch 1/10 | Iteration 600 | loss=1.7558 | val_acc=42.30%
Epoch 1/10 | Iteration 700 | loss=1.5358 | val_acc=44.60%
End of epoch 1: val_acc=43.60%
Epoch 2/10 | Iteration 0 | loss=2.0475 | val_acc=44.00%
Epoch 2/10 | Iteration 100 | loss=1.3665 | val_acc=44.90%
Epoch 2/10 | Iteration 200 | loss=1.4387 | val_acc=47.80%
Epoch 2/10 | Iteration 300 | loss=1.3506 | val_acc=47.30%
Epoch 2/10 | Iteration 400 | loss=1.6671 | val_acc=50.00%
Epoch 2/10 | Iteration 500 | loss=1.5395 | val_acc=48.90%
Epoch 2/10 | Iteration 600 | loss=1.7039 | val_acc=49.70%
Epoch 2/10 | Iteration 700 | loss=1.5296 | val_acc=49.00%
End of epoch 2: val_acc=47.90%
Epoch 3/10 | I

0.5557

## Test set -- run this only once

Now that we've gotten a result we're happy with, we test our final model on the test set (which you should store in best_model). Think about how this compares to your validation set accuracy.

In [19]:
# (Optional) Quick reproducibility check: load the saved best checkpoint and evaluate on test
ckpt_path = os.path.join(CHECKPOINT_DIR, f"betternet_seed{SEED}.pth")

checkpoint = torch.load(ckpt_path, map_location=device)
loaded_model = BetterNet()
loaded_model.load_state_dict(checkpoint["state_dict"])
loaded_model = loaded_model.to(device=device)

print(f"Loaded checkpoint (seed={checkpoint.get('seed')}, best_val_acc={100*checkpoint.get('best_val_acc', 0):.2f}%)")
check_accuracy_part34(loader_test, loaded_model)

Loaded checkpoint (seed=42, best_val_acc=58.60%)
Checking accuracy on test set
Got 5557 / 10000 correct (55.57)


0.5557